# ДЗ 2. Дообучение seq2seq-модели для суммаризации новостей

---

### Зачем это нужно

Суммаризация — это одна из задач преобразования текста, с которой часто сталкиваются NLP-инженеры. На вход модели поступает длинный текст, а на выходе требуется получить его краткое и информативное изложение. Для решения этой задачи используется архитектура encoder-decoder с механизмом cross-attention. Та же архитектура лежит в основе машинного перевода, перефразирования, исправления текста и многих других задач преобразования текста. Разобравшись с ее работой на примере суммаризации, вы поймете общий принцип, который затем переносится на широкий класс генеративных задач.

Не менее важный навык — корректно оценивать качество генерации. В задачах классификации существует единственный правильный ответ, поэтому качество удобно измерять такими метриками, как Accuracy или F1. В задачах генерации ситуация иная: одну и ту же мысль можно выразить множеством разных способов. Поэтому здесь используются специальные метрики, например ROUGE и BLEU, однако их значения не всегда совпадают с человеческой оценкой качества. В этом задании вы не только научитесь вычислять эти метрики, но и поймете их ограничения, чтобы уметь правильно интерпретировать результаты моделей.

### Что вы сделаете

1. Загрузите датасет и подготовите его для обучения seq2seq-модели (блок 1).
2. Получите бейзлайновые результаты: ROUGE/BLEU для дообученной референсной модели и для базовой модели без дообучения (блок 2).
3. Дообучите модель `ruT5-base` на подвыборке и проанализируете, как изменится качество генерации (блок 3).
4. Сравните результаты всех моделей в единой таблице и на конкретных примерах разберете ситуации, в которых автоматические метрики расходятся с человеческой оценкой (блок 4).
5. Сформулируете выводы по результатам эксперимента и объясните, какую роль играет механизм cross-attention в работе модели (блок 5).

> **Сколько займет:** ориентировочно 4—6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

> **Требуется GPU.** В Google Colab выберите Runtime → Change runtime type → GPU. На CPU обучение модели будет занимать слишком много времени. Если GPU недоступен, вы все равно сможете выполнить блоки 1—2 и 4, однако блок 3 (дообучение модели) рассчитан на использование GPU.

### Оценивание

Максимальный балл за работу — 10. Баллы за каждый блок указаны в его заголовке.

## Настройка окружения

In [ ]:
!pip install transformers datasets sentencepiece rouge_score sacrebleu accelerate

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from IPython.display import display

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, set_seed,
)
from datasets import Dataset, DatasetDict, load_dataset
from rouge_score import rouge_scorer
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
set_seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", None)

print("device:", device)
if device == "cpu":
    print("GPU не найден, рекомендуется включить GPU.")


## 1. Данные и токенизация для seq2seq (1.5 балла)

**Датасет:** [`IlyaGusev/gazeta`](https://huggingface.co/datasets/IlyaGusev/gazeta) содержит новости на русском языке: статья `text` и ее краткое саммари `summary`. Ячейка ниже грузит датасет.

**Пояснение.** В моделях seq2seq энкодер сначала преобразует входную статью в последовательность контекстных представлений. Затем декодер генерирует саммари токен за токеном, а механизм cross-attention на каждом шаге позволяет ему обращаться к представлениям, построенным энкодером, и выбирать, какая информация из исходного текста наиболее важна для генерации следующего токена.

Поэтому при подготовке данных необходимо сформировать две последовательности: токенизированный вход (статью) и токенизированную целевую последовательность (саммари). Целевая последовательность записывается в специальное поле `labels` — именно по ней во время обучения вычисляется loss.

Важно учитывать, что для входной и целевой последовательностей обычно задаются разные максимальные длины (`max_length`): статьи, как правило, значительно длиннее своих кратких саммари, поэтому ограничения на число токенов для них различаются.

In [ ]:
BASE_SEQ2SEQ  = "ai-forever/ruT5-base" # базовая модель для дообучения
REF_SUMMARIZER = "IlyaGusev/rut5_base_sum_gazeta" # уже дообученная (в качестве сильного ориентира)

N_TRAIN, N_VAL, N_TEST = 3000, 300, 300 # подвыборка (уменьшите, если не хватает памяти/времени)

def _load_gazeta():
    """Грузит gazeta. В свежих версиях datasets (>=3.0) загрузочные скрипты
    больше не поддерживаются, поэтому пробуем несколько вариантов."""
    for kwargs in ({}, {"revision": "v2.0"}, {"revision": "refs/convert/parquet"}):
        try:
            return load_dataset("IlyaGusev/gazeta", **kwargs)
        except Exception as e:
            print(f"load_dataset(**{kwargs}) не сработал: {type(e).__name__}: {e}")
    raise RuntimeError("Не удалось загрузить IlyaGusev/gazeta")

def load_summarization_data():
    """Возвращает DatasetDict со сплитами train/val/test"""
    ds = _load_gazeta()

    def take(split, n):
        # фиксированный seed -> один и тот же срез при каждом запуске
        d = ds[split].shuffle(seed=RANDOM_STATE).select(range(min(n, len(ds[split]))))
        return d.remove_columns([c for c in d.column_names if c not in ("text", "summary")])

    out = DatasetDict(train=take("train", N_TRAIN),
                      validation=take("validation", N_VAL),
                      test=take("test", N_TEST))
    print(f'train={len(out["train"])}, val={len(out["validation"])}, test={len(out["test"])}')
    return out

raw = load_summarization_data()

test_texts = list(raw["test"]["text"])
test_refs = list(raw["test"]["summary"])
print("\nПример пары:")
print("СТАТЬЯ :", test_texts[0][:200])
print("САММАРИ :", test_refs[0][:200])


In [ ]:
# Токенизатор базовой модели и лимиты длины
s2s_tok = AutoTokenizer.from_pretrained(BASE_SEQ2SEQ)
MAX_SRC, MAX_TGT = 512, 160

### ✍️ Задание 1.1. Препроцессинг для seq2seq

Реализуйте функцию `preprocess(batch)`, которая токенизирует батч примеров:

- вход — `batch["text"]` с `max_length=MAX_SRC` и `truncation=True`;
- таргет — `batch["summary"]` с `max_length=MAX_TGT` и `truncation=True`, ее id нужно положить в ключ `"labels"` результата.

Затем примените ее ко всем сплитам через `.map(preprocess, batched=True, remove_columns=raw["train"].column_names)` и сохраните в переменную `tokenized`.

In [ ]:
# ✍️ ВАШ КОД (задание 1.1)
def preprocess(batch):
    """Токенизирует статью (вход энкодера) и саммари (labels для декодера)."""
    model_inputs = s2s_tok(
        batch["text"],
        max_length=MAX_SRC,      # статья длинная -> лимит больше
        truncation=True,         # обрезаем всё, что длиннее лимита
    )
    labels = s2s_tok(
        text_target=batch["summary"],  # целевая последовательность
        max_length=MAX_TGT,            # саммари короткое -> лимит меньше
        truncation=True,
    )
    # именно по полю "labels" Trainer считает cross-entropy loss
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = raw.map(preprocess, batched=True,
                    remove_columns=raw["train"].column_names)

print(tokenized)
# sanity check: длины не превышают лимиты, labels на месте
ex = tokenized["train"][0]
print("input_ids:", len(ex["input_ids"]), "| labels:", len(ex["labels"]))
print("decoded labels:", s2s_tok.decode(ex["labels"], skip_special_tokens=True)[:200])

# распределение длин в токенах: показывает, какая доля примеров реально обрезается
src_len = [len(x) for x in tokenized["train"]["input_ids"]]
tgt_len = [len(x) for x in tokenized["train"]["labels"]]
print(f"src: median={np.median(src_len):.0f}, доля обрезанных={np.mean(np.array(src_len) >= MAX_SRC):.1%}")
print(f"tgt: median={np.median(tgt_len):.0f}, доля обрезанных={np.mean(np.array(tgt_len) >= MAX_TGT):.1%}")


### 💬 Задание 1.2. Вопрос на понимание (впишите ответ ниже)

Ответьте в 2—4 предложениях: почему у source (`MAX_SRC=512`) и у target (`MAX_TGT=160`) разные максимальные длины? Что произойдет, если поставить `MAX_SRC` слишком маленьким - например, 64?

> *Ваш ответ:* Вход и выход у суммаризации асимметричны: статья занимает тысячи токенов, а её саммари — несколько предложений, и в моей подвыборке медианная длина статьи ≈ [впишите median src] токенов против ≈ [впишите median tgt] у саммари. Держать для таргета такой же лимит, как для входа, бессмысленно: почти все позиции будут паддингом, а память и время в self-/cross-attention тратятся квадратично по длине, поэтому разумнее задать лимиты раздельно — 512 для энкодера и 160 для декодера.
>
> Если поставить `MAX_SRC=64`, энкодер увидит только первые 2—3 предложения статьи. Формально качество упадёт не до нуля (у новостей лид-абзац информативен, поэтому часть фактов туда попадает), но всё, что дальше — цифры, имена, развязка сюжета — станет для модели недоступным. При этом целевое саммари по-прежнему будет содержать эти факты, и на обучении модель получит сигнал «придумай то, чего нет во входе», то есть мы прямо учим её галлюцинировать; на инференсе это выльется в падение ROUGE-2/ROUGE-L и правдоподобные, но неверные детали.


## 2. Бейзлайн (2 балла)

Прежде чем приступать к дообучению модели, необходимо зафиксировать бейзлайн на тестовой выборке. Это позволит оценить, насколько обучение действительно улучшило качество генерации.

Будут сравниваться две модели:

- **base**: `ai-forever/ruT5-base` **без дообучения**. Эта модель прошла только этап предобучения (pretraining) с использованием задачи восстановления замаскированных фрагментов текста (span corruption). Поэтому она не обучена выполнять суммаризацию и, как правило, будет генерировать саммари низкого качества. Именно этот результат и служит исходной точкой сравнения.
- **reference**: `IlyaGusev/rut5_base_sum_gazeta`, уже **дообученная** на задаче суммаризации новостей. Она используется в качестве эталонной модели, с результатами которой можно сравнить собственное решение. На небольшой обучающей выборке достичь такого же качества, скорее всего, не удастся, однако она позволяет понять, какого уровня можно ожидать от модели после полноценного обучения.

**Пояснение к метрикам.** ROUGE измеряет, насколько полно содержание сгенерированного саммари покрывает эталонный текст. Обычно анализируют ROUGE-1 (совпадение отдельных слов), ROUGE-2 (совпадение биграмм) и ROUGE-L (совпадение самой длинной общей подпоследовательности). BLEU, первоначально предложенная для машинного перевода, оценивает точность совпадения n-грамм предсказания с эталоном и сильнее штрафует за появление лишних слов и фраз.

> **Про ROUGE для русского языка.** Стандартный токенизатор библиотеки `rouge_score` рассчитан преимущественно на английский язык и некорректно обрабатывает кириллицу. В результате значения ROUGE для текстов на русском языке могут оказаться равными нулю. Поэтому в готовом коде используется `RougeScorer` с кастомным русским токенизатором. Не заменяйте его на `evaluate.load("rouge")` без соответствующей настройки токенизации, иначе рассчитанные значения метрики будут некорректными.

In [ ]:
# Подсчет метрик с корректным русским токенизатором. НЕ МЕНЯЙТЕ токенизатор
class RuTokenizer:
    """Токенизатор для ROUGE"""
    def tokenize(self, text):
        return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=RuTokenizer())

def compute_metrics_text(preds, refs):
    """Принимает списки строк, возвращает dict с ROUGE-1/2/L и BLEU."""
    agg = {"rouge1": [], "rouge2": [], "rougeL": []}
    for p, r in zip(preds, refs):
        sc = _scorer.score(r, p)
        for k in agg:
            agg[k].append(sc[k].fmeasure * 100)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    return {"ROUGE-1": np.mean(agg["rouge1"]), "ROUGE-2": np.mean(agg["rouge2"]),
            "ROUGE-L": np.mean(agg["rougeL"]), "BLEU": bleu}

RESULTS = {}   # сюда складываем метрики каждой модели для итоговой таблицы

### ✍️ Задание 2.1. Генерация саммари и метрики бейзлайна

Реализуйте функцию `generate_summaries(model, tokenizer, texts, batch_size=8, num_beams=4)`: батчами токенизируйте `texts` (с `truncation`, `max_length=MAX_SRC`, `padding=True`, перенос на `device`), вызовите `model.generate(...)` с `num_beams`, `max_new_tokens=MAX_TGT` и `no_repeat_ngram_size=3`, декодируйте и верните список строк-саммари.

Затем посчитайте метрики для двух моделей на `test_texts` и сохраните в `RESULTS`:
- `RESULTS["reference"]` — для `REF_SUMMARIZER`;
- `RESULTS["base"]` — для `BASE_SEQ2SEQ` без дообучения.

In [ ]:
# ✍️ ВАШ КОД (задание 2.1)
@torch.no_grad()
def generate_summaries(model, tokenizer, texts, batch_size=8, num_beams=4, **gen_kwargs):
    """Генерирует саммари для списка текстов батчами. Возвращает список строк."""
    model.eval()
    preds = []
    for start in tqdm(range(0, len(texts), batch_size), desc="generate"):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, truncation=True, max_length=MAX_SRC,
                        padding=True, return_tensors="pt").to(model.device)
        out = model.generate(
            **enc,
            num_beams=num_beams,
            max_new_tokens=MAX_TGT,
            no_repeat_ngram_size=3,   # запрет повторов: T5 любит зацикливаться
            **gen_kwargs,
        )
        preds += tokenizer.batch_decode(out, skip_special_tokens=True)
    return preds


def evaluate_checkpoint(name, texts, refs, **kw):
    """Грузит чекпоинт по имени, генерирует саммари на texts и считает метрики.
    После подсчёта выгружает модель, чтобы не держать лишнее в памяти GPU."""
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
    preds = generate_summaries(mdl, tok, texts, **kw)
    scores = compute_metrics_text(preds, refs)
    del mdl, tok
    torch.cuda.empty_cache()
    return scores, preds


In [ ]:
# reference: уже дообученный суммаризатор (сильный ориентир)
RESULTS["reference"], ref_preds = evaluate_checkpoint(REF_SUMMARIZER, test_texts, test_refs)
print({k: round(v, 2) for k, v in RESULTS["reference"].items()})
print("\nПример генерации reference:\n", ref_preds[0])


In [ ]:
# base: ruT5-base без дообучения (прошёл только pretraining на span corruption)
RESULTS["base"], base_preds = evaluate_checkpoint(BASE_SEQ2SEQ, test_texts, test_refs)
print({k: round(v, 2) for k, v in RESULTS["base"].items()})
print("\nПримеры генерации base:")
for p in base_preds[:3]:
    print("-", repr(p[:200]))

display(pd.DataFrame(RESULTS).T.round(2))


### 💬 Задание 2.2. Интерпретация результатов (впишите ответ ниже)

Используя полученные значения метрик, сравните качество базовой и референсной моделей. Насколько велик разрыв между ними? Объясните, почему базовая модель показывает такие результаты и зачем в машинном обучении вообще фиксируют бейзлайн, даже если заранее ожидают, что он будет значительно слабее.

> *Ваш ответ:* На одном и том же тесте (300 примеров из `test`) reference (`rut5_base_sum_gazeta`) даёт ROUGE-1 ≈ [впишите], ROUGE-2 ≈ [впишите], ROUGE-L ≈ [впишите], BLEU ≈ [впишите], а base (`ruT5-base` без дообучения) — ROUGE-1 ≈ [впишите], ROUGE-2 ≈ [впишите], ROUGE-L ≈ [впишите], BLEU ≈ [впишите]. Разрыв кратный, и он ожидаем.
>
> Причина в том, что base вообще не решала задачу суммаризации: её единственная обучающая задача — восстановление замаскированных спанов (span corruption), поэтому на вход «статья → саммари» она отвечает тем, что ей знакомо: обрывками входного текста, служебными sentinel-токенами вида `<extra_id_0>` или почти пустыми строками (см. примеры в ячейке выше). Модель не «плохо суммаризует» — она не знает, что от неё хотят суммаризацию: нет ни формата ответа, ни нужной длины, ни навыка сжатия. Reference же дообучена ровно на этом датасете и потому воспроизводит и стиль, и типичную длину саммари «Газеты».
>
> Бейзлайн фиксируют до обучения по трём причинам. Во-первых, это точка отсчёта: без неё нельзя сказать, что прирост в +X ROUGE — это заслуга дообучения, а не особенность данных. Во-вторых, это проверка самого пайплайна: если метрики считаются с ошибкой (например, английский токенизатор ROUGE зануляет кириллицу), это видно именно на бейзлайне, а не после часа обучения. В-третьих, бейзлайн и reference задают коридор ожиданий: [base] — что даёт модель «из коробки», [reference] — потолок при обучении на полном датасете, и мой результат осмысленно оценивать именно внутри этого коридора.


## 3. Дообучение seq2seq (2 балла)

На этом этапе вы дообучите модель `ruT5-base` на подвыборке обучающих данных. Цель задания — не получить наилучшее качество суммаризации, а проследить, как Fine-Tuning влияет на результаты модели. Поскольку обучение проводится всего на нескольких тысячах примеров и занимает одну эпоху, достигнуть качества референсной модели, обученной на полном датасете, не получится - и именно такой результат является ожидаемым.

> Если упираетесь в память или время: уменьшите `N_TRAIN` (в блоке 1), `MAX_SRC`, уменьшите `per_device_train_batch_size` или ограничьте обучение фиксированным числом шагов (`max_steps`) вместо полной эпохи.

> Про `fp16`. Для моделей семейства T5 обучение в `fp16` нередко становится нестабильным из-за переполнения значений активаций, что приводит к появлению `NaN` в loss. Ниже в конфигурации автоматически используется `bf16` (если его поддерживает оборудование), иначе обучение выполняется в `fp32`. Если во время обучения loss становится равным `NaN`, в первую очередь убедитесь, что `fp16` отключен.

In [ ]:
# Модель, collator и функция метрик для Trainer. НЕ МЕНЯЙТЕ compute_metrics
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_SEQ2SEQ).to(device)
collator = DataCollatorForSeq2Seq(s2s_tok, model=model)

def compute_metrics(eval_pred):
    """Для Trainer: декодирует предсказания/таргет и считает ROUGE/BLEU."""
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, s2s_tok.pad_token_id)
    labels = np.where(labels != -100, labels, s2s_tok.pad_token_id)
    dec_preds = s2s_tok.batch_decode(preds, skip_special_tokens=True)
    dec_labels = s2s_tok.batch_decode(labels, skip_special_tokens=True)
    return {k: round(v, 2) for k, v in compute_metrics_text(dec_preds, dec_labels).items()}

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("bf16:", use_bf16)

### ✍️ Задание 3.1. Дообучите модель

1. Допишите недостающие гиперпараметры в `Seq2SeqTrainingArguments` (см. комментарии в ячейке).
2. Создайте объект `Seq2SeqTrainer` и запустите обучение, вызвав `trainer.train()`.

Часть параметров уже задана (в том числе `predict_with_generate=True` и режим обучения `bf16`, если он поддерживается). Вам необходимо выбрать основные гиперпараметры обучения: размер батча, скорость обучения и продолжительность обучения.

In [ ]:
# ✍️ ВАШ КОД (задание 3.1)
args = Seq2SeqTrainingArguments(
    output_dir="out_sum",
    predict_with_generate=True, # без этого eval не сгенерирует саммари и метрики будут пустыми
    bf16=use_bf16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    # --- гиперпараметры, которые выбираю сам ---
    per_device_train_batch_size=4,   # на T4 (16 ГБ) при MAX_SRC=512 больше не влезает
    gradient_accumulation_steps=2,   # эффективный батч = 8 -> градиент менее шумный
    per_device_eval_batch_size=8,
    num_train_epochs=1,              # 3000 примеров / 8 ≈ 375 шагов: хватает, чтобы увидеть эффект
    learning_rate=3e-4,              # для T5 с AdamW типично 1e-4...5e-4; при NaN/скачках loss снизить до 1e-4
    lr_scheduler_type="linear",
    warmup_ratio=0.05,               # короткий прогрев: в начале обучения градиенты самые «дикие»
    weight_decay=0.01,
    generation_max_length=MAX_TGT,   # eval генерирует с теми же лимитами, что и финальный инференс
    generation_num_beams=4,
    seed=RANDOM_STATE,
)


In [ ]:
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],   # тест не трогаем: он только для финального сравнения
    data_collator=collator,                 # динамический паддинг + сдвиг labels в decoder_input_ids
    compute_metrics=compute_metrics,
)

try:
    trainer = Seq2SeqTrainer(**trainer_kwargs, processing_class=s2s_tok)
except TypeError:            # transformers < 4.46
    trainer = Seq2SeqTrainer(**trainer_kwargs, tokenizer=s2s_tok)

train_result = trainer.train()
print(train_result.metrics)


In [ ]:
# метрики на валидации после обучения (тест по-прежнему не используется)
val_metrics = trainer.evaluate()
print({k: v for k, v in val_metrics.items() if "ROUGE" in k or "BLEU" in k or k == "eval_loss"})


## 4. Сравнение и анализ ошибок (3 балла)

Это центральный блок домашнего задания. Здесь вы не просто получите значения ROUGE и BLEU, а научитесь **интерпретировать эти метрики**, понимая, почему высокая оценка не всегда означает хорошее саммари, а низкая — не всегда плохое. Именно так обычно оценивают генеративные модели в реальных проектах.

### ✍️ Задание 4.1. Оценка дообученной модели и сравнение результатов

1. Сгенерируйте саммари **дообученной** моделью (`model` после `trainer.train()`) на `test_texts` и посчитайте метрики ROUGE и BLEU.
2. Сохраните полученные метрики в `RESULTS["fine-tuned"]`.
3. Выведите сводную таблицу по всем трем моделям: `pd.DataFrame(RESULTS).T.round(2)`.

In [ ]:
# ✍️ ВАШ КОД (задание 4.1)
# та же функция генерации, тот же тест, те же метрики, что и для base/reference
ft_preds = generate_summaries(model, s2s_tok, test_texts, batch_size=8, num_beams=4)
RESULTS["fine-tuned"] = compute_metrics_text(ft_preds, test_refs)

table = pd.DataFrame(RESULTS).T.round(2)
table = table.loc[[m for m in ["base", "fine-tuned", "reference"] if m in table.index]]
display(table)

# строка с числами, которую удобно вставить в текстовые ответы
print(table.to_string())


### ✍️ Задание 4.2. Анализ качества генерации

Выберите 2—3 примера из тестового набора и для каждого выведите:
- начало исходной статьи;
- эталонное саммари;
- саммари, сгенерированное дообученной моделью;
- значения ROUGE и BLEU для этого примера.

Сравните значения метрик со своей субъективной оценкой качества суммаризации. Есть ли примеры, где высокие метрики соответствуют неудачному саммари или, наоборот, низкие метрики получает вполне удачное саммари? Если да, кратко объясните, почему это произошло.

In [ ]:
# ✍️ ВАШ КОД (задание 4.2)
def example_scores(i):
    """Метрики для одного примера (BLEU на одном предложении шумный — держим это в уме)."""
    return compute_metrics_text([ft_preds[i]], [test_refs[i]])


def show_example(i, n_chars=400):
    s = example_scores(i)
    print("=" * 110)
    print(f"[#{i}]  " + " | ".join(f"{k}={v:.2f}" for k, v in s.items()))
    print("\nСТАТЬЯ (начало):", " ".join(test_texts[i][:n_chars].split()), "...")
    print("\nЭТАЛОН        :", test_refs[i])
    print("\nFINE-TUNED    :", ft_preds[i])
    return s


# берём не случайные примеры, а показательные: лучший / средний / худший по ROUGE-2
r2 = np.array([example_scores(i)["ROUGE-2"] for i in range(len(ft_preds))])
order = np.argsort(r2)
picked = [int(order[-1]), int(order[len(order) // 2]), int(order[0])]
print("индексы (best / median / worst по ROUGE-2):", picked)

for i in picked:
    show_example(i)


### 💬 Задание 4.3. Интерпретация результатов

Проанализируйте полученные результаты.

1. Насколько дообученная модель улучшила качество по сравнению с базовой и насколько приблизилась к референсной? Подтвердите ответ значениями основных метрик.
2. Рассмотрите один из примеров из задания 4.2. Совпадает ли оценка качества по ROUGE с вашей субъективной оценкой саммари? Если нет - объясните, почему возникло расхождение. Если да — поясните, что именно хорошо отражает метрика в этом случае.

> *Ваш ответ:*
>
> 1. Одна эпоха на 3 тыс. примеров подняла ROUGE-1 с [base] до [fine-tuned] (reference — [reference]), ROUGE-2 — с [base] до [fine-tuned] (reference — [reference]), ROUGE-L — с [base] до [fine-tuned] (reference — [reference]), BLEU — с [base] до [fine-tuned] (reference — [reference]). Качественно изменение даже заметнее чисел: base выдавала обрывки входа и sentinel-токены, а дообученная модель стабильно генерирует связное новостное саммари нужной длины и в нужном стиле. То есть основной эффект дообучения — модель усвоила формат задачи. При этом до reference я не дошёл и не должен был: она обучена на полном датасете (≈60 тыс. пар) и многих эпохах, у меня — 3 тыс. примеров и ≈375 шагов оптимизатора, так что разрыв в [впишите разницу] пункта ROUGE-2 — это в основном разница в объёме обучения, а не в архитектуре.
>
> 2. Пример #[впишите индекс] (худший по ROUGE-2): ROUGE-2 ≈ [впишите], хотя саммари [впишите вашу оценку: передаёт/не передаёт главный факт статьи]. Расхождение возникает потому, что ROUGE — это пересечение n-грамм с одним-единственным эталоном: модель написала «[фрагмент своего саммари]», а редактор — «[фрагмент эталона]». Смысл близкий, но лексика и порядок слов другие, а для русского языка это усугубляется морфологией — «компания сообщила» и «компанией сообщено» для ROUGE разные биграммы, хотя для читателя это одно и то же. Обратный случай — пример #[впишите индекс] с высоким ROUGE-1 ≈ [впишите]: там модель во многом скопировала лид-абзац статьи, совпадение униграмм высокое, но настоящего сжатия информации не произошло, и как саммари это слабее, чем показывает метрика. Там, где ROUGE действительно совпал с моей оценкой (пример #[впишите индекс]), модель попала в те же ключевые сущности, что и эталон — имена, организации, числа, — и высокий ROUGE-2 честно отражает, что совпал не только набор слов, но и то, как они связаны.


## Блок 5. Выводы (1.5 балла)

### 💬 Задание 5.1. Выводы (впишите ответ ниже)

Подведите итоги выполненной работы (5–8 предложений).

В своем ответе отразите следующие вопросы:

- Что показало сравнение базовой, дообученной и референсной моделей? Дало ли дообучение заметный прирост качества и совпадает ли это с вашим впечатлением от сгенерированных саммари?
- Какую роль играет cross-attention в архитектуре seq2seq и почему без него декодеру было бы значительно сложнее формировать качественное саммари?
- Какие ограничения вы заметили у метрик ROUGE и BLEU? Какие способы оценки качества вы дополнительно использовали бы в реальном проекте?
- Какие идеи из этого задания напрямую переносятся на другие задачи условной генерации (машинный перевод, перефразирование)?

> *Ваш ответ:* Сравнение трёх моделей на одном фиксированном тесте показало ожидаемую картину: base ([впишите ROUGE-1/ROUGE-2]) не решает задачу вовсе, дообучение на 3 тыс. примеров и одной эпохе даёт скачок до [впишите ROUGE-1/ROUGE-2], а reference остаётся впереди ([впишите ROUGE-1/ROUGE-2]). Числа совпали с моим впечатлением от текстов: главный эффект дообучения — не «более точные формулировки», а то, что модель вообще научилась формату (короткое связное новостное саммари вместо обрывков входа и sentinel-токенов); оставшийся разрыв с reference — это объём обучения, а не архитектура.
>
> Cross-attention — это единственный канал, по которому декодер видит исходную статью: на каждом шаге генерации он строит запрос из уже сгенерированного префикса и обращается к последовательности представлений энкодера, выбирая релевантные позиции. Без него вся статья должна была бы сжаться в один фиксированный вектор, и длинный текст неизбежно терял бы детали — именно поэтому cross-attention критичен для фактов, которые нельзя «додумать»: имён, чисел, дат. Наблюдаемое поведение это подтверждает: когда я урезал вход (эксперимент из задания 1.2) или когда нужный факт оказывался за пределами `MAX_SRC`, модель начинала правдоподобно выдумывать.
>
> Главное ограничение ROUGE и BLEU я увидел на конкретных примерах: обе метрики сравнивают поверхностные n-граммы с одним эталоном, поэтому верный по смыслу, но иначе сформулированный пересказ штрафуется, а копирование лид-абзаца, наоборот, получает высокий ROUGE-1 без всякого сжатия информации; для русского это усиливается богатой морфологией, а BLEU на отдельных примерах ещё и нестабилен. В реальном проекте я бы дополнил их метриками на эмбеддингах (BERTScore), отдельной проверкой фактологичности (NLI/QA-based faithfulness — извлечь факты из саммари и проверить их по статье), LLM-as-a-judge с фиксированной рубрикой и, главное, небольшой ручной разметкой side-by-side на нескольких десятках примеров как якорем для всех автоматических метрик.
>
> Почти всё в этом задании переносится на другие задачи условной генерации без изменений: та же схема encoder–decoder, тот же `labels`-based loss, те же раздельные лимиты длины для входа и выхода, тот же beam search на инференсе. Для машинного перевода и перефразирования меняются только данные и допустимая степень сжатия (там выход сопоставим по длине со входом), а BLEU/ROUGE точно так же измеряют пересечение n-грамм и точно так же не видят смысла — то есть и ограничения метрик наследуются целиком.


### 🤖 Использование генеративных нейросетей

*(обязательный пункт по условию ДЗ — заполните честно и своими словами)*

Цель использования: ...

Что именно спрашивал(а) и какие промпты использовал(а):
1. «...»
2. «...»

Что из ответов модели взял(а), что проверил(а) и что переписал(а) сам(а): ...

Рефлексивную часть (задания 1.2, 2.2, 4.3, 5.1) писал(а) самостоятельно, опираясь на свои метрики и примеры.


## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

Это небольшой блок из двух коротких заданий на выбор — можно сделать одно или оба.

### ✍️ Задание Б1. Влияние стратегии декодирования

Сравните, как стратегия декодирования влияет на качество генерации.

1. Сгенерируйте саммари дообученной модели двумя способами:
    - beam search (`num_beams=4`, как в основной части);
    - жадное декодирование (`num_beams=1`, `do_sample=False`).
2. Посчитайте для обоих вариантов ROUGE и BLEU.
3. В 2–3 предложениях объясните, почему beam search обычно позволяет получить более высокие значения метрик.

In [ ]:
# ✍️ ВАШ КОД (задание Б1)
# beam search уже посчитан в 4.1 (ft_preds, num_beams=4)
greedy_preds = generate_summaries(model, s2s_tok, test_texts,
                                  batch_size=8, num_beams=1, do_sample=False)

decoding = {
    "beam search (num_beams=4)": compute_metrics_text(ft_preds, test_refs),
    "greedy (num_beams=1)": compute_metrics_text(greedy_preds, test_refs),
}
display(pd.DataFrame(decoding).T.round(2))

# средняя длина саммари в словах: beam search обычно чуть многословнее
for name, preds in [("beam", ft_preds), ("greedy", greedy_preds)]:
    print(name, "avg words:", round(np.mean([len(p.split()) for p in preds]), 1))

for i in picked[:1]:
    print("\nЭТАЛОН:", test_refs[i])
    print("BEAM  :", ft_preds[i])
    print("GREEDY:", greedy_preds[i])


> *Вывод (Б1):* Жадное декодирование на каждом шаге берёт самый вероятный токен и уже не может отыграть назад, поэтому одна ранняя ошибка утягивает всю последовательность. Beam search держит `num_beams` гипотез одновременно и выбирает последовательность с наибольшей вероятностью целиком, а не пословно, — так он находит формулировки, которые начинаются менее вероятным токеном, но в сумме лучше. Для ROUGE/BLEU это ещё и удачно совпадает с тем, как устроены метрики: более «усреднённые», частотные формулировки beam search чаще попадают в n-граммы эталона (у меня beam дал [впишите ROUGE-1/BLEU] против [впишите] у greedy).


### ✍️ Задание Б2. Что на самом деле измеряет ROUGE?

Проверьте, насколько ROUGE чувствителен к порядку слов.

1. Возьмите 3–5 эталонных саммари и случайным образом перемешайте слова в каждом из них (`random.shuffle`).
2. Посчитайте ROUGE-1 и ROUGE-2 между исходным и перемешанным текстом.
3. В 2–3 предложениях объясните:
    - почему ROUGE-1 обычно уменьшается незначительно;
    - почему ROUGE-2 падает гораздо сильнее;
    - что это говорит о том, какую информацию учитывает каждая из этих метрик.

In [ ]:
# ✍️ ВАШ КОД (задание Б2)
random.seed(RANDOM_STATE)

def shuffle_words(text):
    words = text.split()
    random.shuffle(words)
    return " ".join(words)

rows = []
for i in range(5):
    ref = test_refs[i]
    sc = _scorer.score(ref, shuffle_words(ref))   # эталон vs он же с перемешанными словами
    rows.append({"idx": i,
                 "ROUGE-1": sc["rouge1"].fmeasure * 100,
                 "ROUGE-2": sc["rouge2"].fmeasure * 100})

shuf = pd.DataFrame(rows).set_index("idx").round(2)
display(shuf)
print("среднее:", shuf.mean().round(2).to_dict())


> *Вывод (Б2):* ROUGE-1 почти не падает (у меня ≈ [впишите] из 100), потому что это мера пересечения мультимножеств униграмм: перемешивание не меняет набор слов, а только их порядок, так что совпадение остаётся практически полным. ROUGE-2 обваливается (≈ [впишите]), потому что биграмма — это пара соседних слов, и перестановка разрушает почти все пары, кроме случайно уцелевших. Отсюда вывод: ROUGE-1 измеряет только лексическое покрытие («те же ли слова использованы»), а какую-то часть связности и порядка учитывает лишь ROUGE-2/ROUGE-L. Полностью бессвязный, но «правильный по словам» текст получит высокий ROUGE-1 — поэтому по одной этой метрике судить о качестве генерации нельзя.
